
## Architecture Position

***The ETL Control Framework sits alongside the Bronze-to-Silver processing
flow. It does not contain insurance business data; it controls and records
how incremental processing executes.***


```text
                         ┌─────────────────────────────┐
                         │      ETL CONTROL FRAMEWORK  │
                         │                             │
                         │  etl_control                │
                         │  • source configuration     │
                         │  • watermark column         │
                         │  • last successful watermark│
                         │                             │
                         │  etl_batch_audit            │
                         │  • batch ID                 │
                         │  • inserts / updates        │
                         │  • status / errors          │
                         └──────────────┬──────────────┘
                                        │
                          controls + audits
                                        │
                                        ▼
┌─────────────────┐       ┌──────────────────────────┐       ┌─────────────────┐
│    LH_Bronze    │       │ Incremental Processing   │       │    LH_Silver    │
│                 │       │                          │       │                 │
│ bronze_customers├──────►│ Read watermark           ├──────►│ silver_customers│
│ bronze_policies │       │ Filter changed records   │       │ silver_policies │
│ bronze_claims   │       │ Transform                │       │ silver_claims   │
│ bronze_payments │       │ Delta MERGE              │       │ silver_payments │
└─────────────────┘       │ Update audit/watermark   │       └─────────────────┘
                          └──────────────────────────┘


**Data flow**

`LH_Bronze → Incremental Processing → LH_Silver`

**Control flow**

`etl_control → incremental processor → new watermark`

**Observability**

`incremental processor → etl_batch_audit`

That distinction is very important architecturally.

Also, keep your existing **Purpose** section. It is already good.

After this architecture section, keep the sections we created for **Tables Created**, **Fabric Components Used**, and **Processing Pattern**.

Then the notebook will read like an actual engineering artifact rather than just a collection of Spark cells.

Once you've replaced that section, we're ready for **Code Cell 1 — imports and framework configuration**.



## Step 1 — Initialize the ETL Control Framework

### What this step does

This cell imports the Spark and Delta Lake functionality required by the
control framework and defines the location of the operational metadata tables.

The framework will maintain two Delta tables in the Silver Lakehouse:

- `etl_control` — controls incremental processing and stores watermarks.
- `etl_batch_audit` — records the execution history of incremental loads.

### Why these tables are in Silver

Bronze represents raw source data.

Silver represents validated and operationally managed data. The control and
audit tables are therefore maintained alongside the Silver processing layer,
rather than mixed with raw Bronze business data.

### Technologies used

- PySpark
- Delta Lake
- Microsoft Fabric Lakehouse
- OneLake

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    BooleanType,
    TimestampType
)
from delta.tables import DeltaTable

# ---------------------------------------------------------
# ETL CONTROL FRAMEWORK CONFIGURATION
# ---------------------------------------------------------

CONTROL_TABLE = "LH_Silver.dbo.etl_control"
AUDIT_TABLE   = "LH_Silver.dbo.etl_batch_audit"

PIPELINE_NAME = "PL_Insurance_Medallion_ETL"

print("ETL Control Framework initialized.")
print(f"Control table : {CONTROL_TABLE}")
print(f"Audit table   : {AUDIT_TABLE}")
print(f"Pipeline      : {PIPELINE_NAME}")


StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 3, Finished, Available, Finished, False)

ETL Control Framework initialized.
Control table : LH_Silver.dbo.etl_control
Audit table   : LH_Silver.dbo.etl_batch_audit
Pipeline      : PL_Insurance_Medallion_ETL



## Step 2 — Create the Incremental Load Control Table

### Purpose

The `etl_control` table stores metadata that tells each incremental
Bronze-to-Silver notebook what data should be processed.

Instead of scanning and reprocessing the complete Bronze table during
every execution, the notebook reads the previous successful watermark
and processes only records newer than that value.

### Example

If the control table contains:

| Source | Watermark Column | Last Watermark |
|---|---|---|
| CLAIMS | last_updated | 2027-10-21 00:00:00 |

the Claims incremental notebook will logically process:

`last_updated > 2027-10-21 00:00:00`

### Control Table Columns

| Column | Purpose |
|---|---|
| source_name | Logical source/entity name |
| source_table | Bronze source table |
| target_table | Silver destination table |
| watermark_column | Source column used for incremental detection |
| load_type | INCREMENTAL or FULL |
| last_watermark | Last successfully processed value |
| is_active | Enables/disables processing for the entity |
| _created_ts | Metadata creation timestamp |
| _updated_ts | Last metadata update timestamp |

### Important Design Principle

The watermark represents the **last successfully processed position**.

It must not be advanced before the Silver write succeeds.

If processing fails, the old watermark remains unchanged so that the
same records can safely be processed again during the next execution.


In [2]:

# ---------------------------------------------------------
# ETL CONTROL TABLE SCHEMA
# ---------------------------------------------------------

control_schema = StructType([
    StructField("source_name",      StringType(),    False),
    StructField("source_table",     StringType(),    False),
    StructField("target_table",     StringType(),    False),
    StructField("watermark_column", StringType(),    False),
    StructField("load_type",        StringType(),    False),
    StructField("last_watermark",   TimestampType(), True),
    StructField("is_active",        BooleanType(),   False),
    StructField("_created_ts",      TimestampType(), False),
    StructField("_updated_ts",      TimestampType(), False)
])

print("etl_control schema defined.")

for field in control_schema.fields:
    print(
        f"{field.name:20} "
        f"{str(field.dataType):20} "
        f"nullable={field.nullable}"
    )
    

StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 4, Finished, Available, Finished, False)

etl_control schema defined.
source_name          StringType()         nullable=False
source_table         StringType()         nullable=False
target_table         StringType()         nullable=False
watermark_column     StringType()         nullable=False
load_type            StringType()         nullable=False
last_watermark       TimestampType()      nullable=True
is_active            BooleanType()        nullable=False
_created_ts          TimestampType()      nullable=False
_updated_ts          TimestampType()      nullable=False



## Step 2B — Create the Physical `etl_control` Delta Table

The previous step defined the schema in Python.

This step creates the actual Delta table in:

`LH_Silver.dbo.etl_control`

The table is initially empty.

We create it before inserting configuration rows so that the schema is
explicit and controlled from the start.

### Why Delta?

Delta Lake gives us transactional table behavior and supports future operations
such as:

- UPDATE
- MERGE
- DELETE

These capabilities are important because the watermark value will change after
successful incremental processing.


In [4]:

# ---------------------------------------------------------
# CREATE EMPTY ETL CONTROL DELTA TABLE
# ---------------------------------------------------------

empty_control_df = spark.createDataFrame(
    [],
    schema=control_schema
)

(
    empty_control_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CONTROL_TABLE)
)

print(f"Created empty Delta table: {CONTROL_TABLE}")

display(
    spark.table(CONTROL_TABLE)
)


StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 6, Finished, Available, Finished, False)

Created empty Delta table: LH_Silver.dbo.etl_control


SynapseWidget(Synapse.DataFrame, ab33c87b-2475-4fd5-acbe-083957a339e3)


## Step 2C — Register Incremental Source Entities

The control table now exists but contains no source configuration.

This step registers the four insurance entities processed from Bronze to Silver:

- CUSTOMERS
- POLICIES
- CLAIMS
- PAYMENTS

Each entity defines:

1. Bronze source table
2. Silver target table
3. Business watermark column
4. Load strategy
5. Initial watermark

### Initial Watermark Strategy

For this project, the initial watermark is:

`1900-01-01 00:00:00`

This represents the starting point before any source data has been processed.

Therefore, the first incremental execution effectively processes all existing
records whose watermark is later than this baseline.

After a successful load, the framework replaces this value with the maximum
successfully processed source watermark.
### Entity Watermarks
| Entity | Bronze Source | Watermark |
|---|---|---|
| CUSTOMERS | bronze_customers | created_date |
| POLICIES | bronze_policies | last_updated |
| CLAIMS | bronze_claims | last_updated |
| PAYMENTS | bronze_payments | payment_date |

> The watermark column does not need to have the same name across all sources.
> The control table makes the incremental framework metadata-driven.


In [5]:
from datetime import datetime

# ---------------------------------------------------------
# REGISTER INCREMENTAL SOURCE ENTITIES
# ---------------------------------------------------------

INITIAL_WATERMARK = datetime(1900, 1, 1)

now_ts = datetime.now()

control_rows = [
    (
        "CUSTOMERS",
        "LH_Bronze.dbo.bronze_customers",
        "LH_Silver.dbo.silver_customers",
        "created_date",
        "INCREMENTAL",
        INITIAL_WATERMARK,
        True,
        now_ts,
        now_ts
    ),
    (
        "POLICIES",
        "LH_Bronze.dbo.bronze_policies",
        "LH_Silver.dbo.silver_policies",
        "last_updated",
        "INCREMENTAL",
        INITIAL_WATERMARK,
        True,
        now_ts,
        now_ts
    ),
    (
        "CLAIMS",
        "LH_Bronze.dbo.bronze_claims",
        "LH_Silver.dbo.silver_claims",
        "last_updated",
        "INCREMENTAL",
        INITIAL_WATERMARK,
        True,
        now_ts,
        now_ts
    ),
    (
        "PAYMENTS",
        "LH_Bronze.dbo.bronze_payments",
        "LH_Silver.dbo.silver_payments",
        "payment_date",
        "INCREMENTAL",
        INITIAL_WATERMARK,
        True,
        now_ts,
        now_ts
    )
]

control_df = spark.createDataFrame(
    control_rows,
    schema=control_schema
)

display(control_df)

StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e70fbbe6-5dbd-4cf2-8e1c-2217ba3b59e6)


## Step 2D — Persist Control Configuration Using Delta MERGE

The configuration records have been validated in memory.

This step persists them into the `etl_control` Delta table.

### Why use MERGE instead of APPEND?

Using `append` would insert another copy of the same configuration every
time this notebook is rerun.

Instead, Delta Lake `MERGE` allows us to make configuration registration
idempotent.

The logical key is:

`source_name`

### MERGE Behavior

If the source already exists:

- Update source table configuration
- Update target table configuration
- Update watermark column
- Update load type
- Update active status
- Preserve the existing `last_watermark`
- Preserve the original `_created_ts`
- Update `_updated_ts`

If the source does not exist:

- Insert a new control record

### Important Watermark Protection

A notebook rerun must **not reset an existing watermark back to
1900-01-01**.

The watermark represents processing state and should only be advanced by
successful incremental processing.


In [6]:

# ---------------------------------------------------------
# UPSERT CONFIGURATION INTO ETL CONTROL
# ---------------------------------------------------------

control_delta = DeltaTable.forName(
    spark,
    CONTROL_TABLE
)

(
    control_delta.alias("target")
    .merge(
        control_df.alias("source"),
        "target.source_name = source.source_name"
    )
    .whenMatchedUpdate(
        set={
            "source_table":     "source.source_table",
            "target_table":     "source.target_table",
            "watermark_column": "source.watermark_column",
            "load_type":        "source.load_type",
            "is_active":        "source.is_active",
            "_updated_ts":      "source._updated_ts"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("ETL control configuration successfully registered.")

StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 8, Finished, Available, Finished, False)

ETL control configuration successfully registered.



***Step 2E — Verify***
--------------
After the MERGE succeeds, run:

In [7]:
control_check_df = (
    spark.table(CONTROL_TABLE)
    .orderBy("source_name")
)

display(control_check_df)

print(
    "Control record count:",
    control_check_df.count()
)


StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a56768f-ef28-48c0-b4b7-3466d67c4780)

Control record count: 4



## Step 3 — Create the ETL Batch Audit Table

### Purpose

The `etl_batch_audit` table provides operational observability for
incremental data processing.

While `etl_control` answers:

> "Where should the next incremental load start?"

`etl_batch_audit` answers:

> "What happened during each execution?"

Each source processed during a pipeline run writes one audit record.

### Example

A single pipeline execution may generate:

| Batch ID | Table | Source | Inserts | Updates | Rejects | Status |
|---|---|---:|---:|---:|---:|---|
| BATCH-001 | CUSTOMERS | 25 | 10 | 15 | 0 | SUCCESS |
| BATCH-001 | POLICIES | 40 | 5 | 35 | 0 | SUCCESS |
| BATCH-001 | CLAIMS | 100 | 20 | 78 | 2 | SUCCESS |
| BATCH-001 | PAYMENTS | 60 | 15 | 45 | 0 | SUCCESS |

The same `batch_id` can therefore connect multiple entity executions
belonging to one pipeline run.

### Audit Table Columns

| Column | Purpose |
|---|---|
| batch_id | Unique identifier for the pipeline execution |
| pipeline_name | Pipeline responsible for the execution |
| table_name | Insurance entity being processed |
| start_time | Processing start timestamp |
| end_time | Processing completion timestamp |
| source_count | Number of incremental source records |
| insert_count | Records inserted into Silver |
| update_count | Existing Silver records updated |
| reject_count | Records rejected by validation |
| status | SUCCESS, FAILED, or NO_DATA |
| error_message | Failure details when processing fails |

### Why This Matters

The audit table provides:

- Operational monitoring
- Troubleshooting
- Record-count reconciliation
- Pipeline execution history
- Failure analysis
- Production support evidence


In [8]:

# ---------------------------------------------------------
# ETL BATCH AUDIT TABLE SCHEMA
# ---------------------------------------------------------

audit_schema = StructType([
    StructField("batch_id",      StringType(),    False),
    StructField("pipeline_name", StringType(),    False),
    StructField("table_name",    StringType(),    False),
    StructField("start_time",    TimestampType(), False),
    StructField("end_time",      TimestampType(), True),
    StructField("source_count",  LongType(),      False),
    StructField("insert_count",  LongType(),      False),
    StructField("update_count",  LongType(),      False),
    StructField("reject_count",  LongType(),      False),
    StructField("status",        StringType(),    False),
    StructField("error_message", StringType(),    True)
])

print("etl_batch_audit schema defined.")

for field in audit_schema.fields:
    print(
        f"{field.name:20} "
        f"{str(field.dataType):20} "
        f"nullable={field.nullable}"
    )

StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 10, Finished, Available, Finished, False)

etl_batch_audit schema defined.
batch_id             StringType()         nullable=False
pipeline_name        StringType()         nullable=False
table_name           StringType()         nullable=False
start_time           TimestampType()      nullable=False
end_time             TimestampType()      nullable=True
source_count         LongType()           nullable=False
insert_count         LongType()           nullable=False
update_count         LongType()           nullable=False
reject_count         LongType()           nullable=False
status               StringType()         nullable=False
error_message        StringType()         nullable=True



## Step 3B — Create the Physical Audit Delta Table

The audit schema has now been defined.

This step creates the physical Delta table:

`LH_Silver.dbo.etl_batch_audit`

The table starts empty. Incremental processing notebooks will write audit
records as they execute.

Unlike `etl_control`, this table does not require seed data because audit
records represent actual processing events.

In [9]:

# ---------------------------------------------------------
# CREATE EMPTY ETL BATCH AUDIT DELTA TABLE
# ---------------------------------------------------------

empty_audit_df = spark.createDataFrame(
    [],
    schema=audit_schema
)

(
    empty_audit_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(AUDIT_TABLE)
)

print(f"Created empty Delta table: {AUDIT_TABLE}")

display(
    spark.table(AUDIT_TABLE)
)

print(
    "Audit record count:",
    spark.table(AUDIT_TABLE).count()
)

StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 11, Finished, Available, Finished, False)

Created empty Delta table: LH_Silver.dbo.etl_batch_audit


SynapseWidget(Synapse.DataFrame, c72a8d3c-4fcc-4968-be77-7b0e3b400df5)

Audit record count: 0



## Step 4 — Validate the ETL Control Framework

Before downstream incremental notebooks use the framework, this step validates
that the metadata environment is correctly configured.

### Validation Checks

The notebook verifies that:

1. `etl_control` exists.
2. `etl_batch_audit` exists.
3. Exactly four source entities are configured.
4. Each `source_name` is unique.
5. All configured entities are active.
6. Every entity has a valid watermark configuration.
7. The audit table is available for downstream execution logging.

### Expected Configuration

| Entity | Watermark Column | Load Type |
|---|---|---|
| CUSTOMERS | created_date | INCREMENTAL |
| POLICIES | last_updated | INCREMENTAL |
| CLAIMS | last_updated | INCREMENTAL |
| PAYMENTS | payment_date | INCREMENTAL |

Successful validation means the control framework is ready for the
Bronze-to-Silver incremental processing notebooks.

In [10]:

# ---------------------------------------------------------
# VALIDATE ETL CONTROL FRAMEWORK
# ---------------------------------------------------------

control_validation_df = spark.table(CONTROL_TABLE)
audit_validation_df   = spark.table(AUDIT_TABLE)

# Basic counts
control_count = control_validation_df.count()
audit_count   = audit_validation_df.count()

# Uniqueness
unique_sources = (
    control_validation_df
    .select("source_name")
    .distinct()
    .count()
)

# Active entities
active_sources = (
    control_validation_df
    .filter(F.col("is_active") == True)
    .count()
)

# Missing critical configuration
invalid_config_count = (
    control_validation_df
    .filter(
        F.col("source_name").isNull()
        | F.col("source_table").isNull()
        | F.col("target_table").isNull()
        | F.col("watermark_column").isNull()
        | F.col("load_type").isNull()
    )
    .count()
)

print("==============================================")
print(" ETL CONTROL FRAMEWORK VALIDATION")
print("==============================================")

print(f"Control records       : {control_count}")
print(f"Unique source names   : {unique_sources}")
print(f"Active sources        : {active_sources}")
print(f"Invalid configurations: {invalid_config_count}")
print(f"Audit records         : {audit_count}")

assert control_count == 4, \
    "Expected exactly 4 control records."

assert unique_sources == control_count, \
    "Duplicate source_name detected."

assert active_sources == 4, \
    "Expected all 4 entities to be active."

assert invalid_config_count == 0, \
    "Invalid control configuration detected."

print()
print("ETL Control Framework validation PASSED.")

StatementMeta(, 55dc9092-621e-4a52-85da-d0e81dc2ad4d, 12, Finished, Available, Finished, False)

 ETL CONTROL FRAMEWORK VALIDATION
Control records       : 4
Unique source names   : 4
Active sources        : 4
Invalid configurations: 0
Audit records         : 0

ETL Control Framework validation PASSED.


---

# Notebook Complete — ETL Control Framework

## What This Notebook Built

This notebook established the metadata-driven control framework used by
the Bronze-to-Silver incremental processing architecture.

### Created Delta Tables

**`LH_Silver.dbo.etl_control`**

Maintains:

- Source and target table configuration
- Watermark column configuration
- Current processing watermark
- Load type
- Active/inactive processing status

**`LH_Silver.dbo.etl_batch_audit`**

Maintains:

- Pipeline execution identifier
- Entity-level execution history
- Source record counts
- Insert and update counts
- Reject counts
- Processing status
- Error information

***Key Design Principles***
1. Processing is metadata-driven rather than hard-coded.
2. Watermarks determine which source records require processing.
3. Delta Lake provides transactional MERGE and update capabilities.
4. Configuration initialization is idempotent.
5. Watermarks are preserved when the framework notebook is rerun.
6. Watermarks advance only after successful downstream processing.
7. Audit records provide operational traceability and reconciliation.

## Processing Architecture

```text
                         etl_control
                              |
                       Read Watermark
                              |
                              v
LH_Bronze  --->  Incremental Processing  --->  LH_Silver
                              |
                    +---------+---------+
                    |                   |
                    v                   v
             etl_batch_audit      Update Watermark



Then **save the notebook**.

---

### Where we are now

This distinction is important:

```text
NB_01_ETL_Control_Framework
          │
          │ metadata/control infrastructure
          │
          ├── etl_control
          └── etl_batch_audit
                    │
                    ▼
         ENTITY PROCESSING NOTEBOOKS
                    │
       ┌────────────┼─────────────┐
       ▼            ▼             ▼
 Customers      Policies       Claims ...